In [ ]:
!pip install w3lib

In [ ]:
pip install lxml requests bs4

In [ ]:
pip install aiohttp

## Load Crawled URLs

This cell loads the list of Ooredoo URLs that were discovered during the crawling step.

The loaded list is used as the input for the scraping pipeline. Each URL will later be fetched, parsed, cleaned, classified, and converted into a structured record.

In [7]:
import json
file_path = r"C:\Users\USER\Desktop\pfe_26\crawled_urls\urls_final.json"

with open(file_path, 'r', encoding='utf-8') as f:
    list_final = json.load(f)
print(len(list_final))

1333


In [8]:
from bs4 import BeautifulSoup
import re
import json
import hashlib
import asyncio
import aiohttp

from dataclasses import dataclass, asdict
from collections import defaultdict
from urllib.parse import urlsplit


# =========================
# Data model
# =========================
@dataclass(frozen=True)
class Record:
    url: str
    title: str
    http_status: int
    section: str
    language: str
    page_type: str
    raw_text: str
    clean_text: str
    embedding_text: str
    content_hash: str
    status: str
    reason: str


# =========================
# Patterns / constants
# =========================
SOFT_404_PATTERNS = [
    r"\b404\b",
    r"page\s+not\s+found",
    r"page\s+introuvable",
    r"n['’]existe\s+pas",
    r"(erreur|error)\s*404",
    r"oops! that link is broken",
]

GLOBAL_REMOVE_SELECTORS = [
    "script", "style", "noscript", "iframe", "svg",
    "header", "footer", "nav", "aside",
    ".cookie", ".cookie-banner", ".cookie-consent",
    ".modal", ".popup", ".dialog",
    ".breadcrumb", ".pagination", ".social-share",
    ".newsletter", ".subscribe",
    "[aria-modal='true']", "[role='dialog']", "[role='alertdialog']",
]

MAIN_SELECTORS = [
    ".product-features", ".product_fatures",
    "main", '[role="main"]', ".main-content", ".page-content",
    "#content", "#main", "article", ".content",
]

TEXT_TAGS = ["h1", "h2", "h3", "p", "li", "td", "th", "dt", "dd"]

ECOM_REMOVE_SELECTORS = [
    ".reviews", ".review", ".rating",
    ".related-products", ".crossselling", ".upsell",
    ".services-eshopping", ".eshop-service-block",
    "[id*='review']", "[class*='review']",
]

NOISE_CONTAINS = [
    "ecrire un commentaire", "écrire un commentaire", "write a review",
    "be the first to write your review",
    "services eshopping", "shopping services",
    "you might also like", "vous aimerez aussi",
    "add to cart", "ajouter au panier",
    "wishlist", "liste de souhaits",
    "compare", "comparer",
    "notify me when available", "prévenez-moi lorsque le produit est disponible",
    "simulateur points merci",
    "j’en profite", "j'en profite",
]

PRODUCT_HERO_SELECTORS = [
    "h1.product-detail-name",
    "h1[itemprop='name']",
    "h1",
    "div.description-short",
    "div[id^='product-description-short-']",
    "div[itemprop='description']",
    ".wbpcity",
    ".wbpdtbn",
    ".product-actions",
]

PRODUCT_KEEP_SELECTORS = [
    "#product-details", ".product-details", ".product-features", ".product_fatures",
    ".tab-content", ".tab-pane", "[id*='detail']", "[class*='detail']",
    "[id*='spec']", "[class*='spec']",
    ".product-information",
]

# Listing/brand specific selectors
LISTING_CONTAINER_SELECTORS = [
    "#js-product-list",
    ".js-product-list",
    ".products",
    ".product-list",
    ".category-products",
]

LISTING_CARD_SELECTORS = [
    "div.ajax_block_product",
    ".product-miniature",
    ".product-item",
    ".js-product",
    ".product-card",
    ".thumbnail-container",
    ".ajax_block_product",
]

PRODUCT_LINK_RE = re.compile(r"\.html(?:$|\?)", re.I)

PRICE_RE = re.compile(r"(\d[\d\s.,]{1,20}\s?(?:DT|TND))", re.I)
POINTS_RE = re.compile(r"(?:\bou\b\s*)?(\d[\d\s.,]{2,20})\s*points?\s*merci", re.I)
STOCK_RE = re.compile(
    r"(stock\s+épuisé|stock\s+epuise|out[- ]of[- ]stock|stock\s+available|stock\s+unavailable|en\s+stock)",
    re.I,
)
COUNT_RE = re.compile(r"\b\d+\s+produits?\b|\b\d+\s+products?\b", re.I)


# =========================
# URL helpers
# =========================
def normalize_url(url: str) -> str:
    u = urlsplit(url.strip())
    return u._replace(fragment="").geturl()


def classify_url(url: str) -> tuple[str, str]:
    path = urlsplit(url).path.lower()
    if "/personal/en" in path:
        return "personal", "en"
    if "/personal/fr" in path:
        return "personal", "fr"
    if "/business/en" in path:
        return "business", "en"
    if "/business/fr" in path:
        return "business", "fr"
    return "unknown", "unknown"


def classify_page_type(url: str) -> str:
    path = urlsplit(url).path.lower()

    if "/content/category/" in path:
        return "category_page"
    if "/brand/" in path:
        return "brand_page"
    if "/content/" in path:
        if "boutiques" in path or "shops" in path:
            return "shop_page"
        return "content_page"
    if path.endswith(".html"):
        return "product_page"
    if re.search(r"/\d+-[a-z0-9-]+$", path):
        return "listing_page"
    if path.endswith("/accueil") or path.endswith("/en") or path.endswith("/fr"):
        return "landing_page"
    return "other"


# =========================
# Utilities
# =========================
def normalize_txt(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in text.split("\n")]
    lines = [ln for ln in lines if ln]
    return "\n".join(lines).strip()


def to_embedding_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\\n", "\n").replace("\\r", "\n").replace("\\t", " ")
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
    text = normalize_txt(text)
    return re.sub(r"\s+", " ", text).strip()


def make_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def element_text(el) -> str:
    return re.sub(r"\s+", " ", el.get_text(" ", strip=True)).strip()


def clean_blocks(lines: list[str]) -> list[str]:
    cleaned = []
    seen = set()
    for line in lines:
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue
        low = line.lower()
        if any(x in low for x in NOISE_CONTAINS):
            continue
        if low in seen:
            continue
        seen.add(low)
        cleaned.append(line)
    return cleaned


# =========================
# Soup helpers
# =========================
def is_soft_404(soup: BeautifulSoup) -> bool:
    probe = " ".join([
        soup.title.get_text(" ", strip=True) if soup.title else "",
        " ".join(h.get_text(" ", strip=True) for h in soup.select("h1"))[:600],
        soup.get_text(" ", strip=True)[:4000],
    ]).lower()
    return any(re.search(p, probe) for p in SOFT_404_PATTERNS)


def get_title(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1 and h1.get_text(" ", strip=True):
        return h1.get_text(" ", strip=True)

    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        return og["content"].strip()

    t = soup.find("title")
    return t.get_text(" ", strip=True) if t else "No title"


def remove_global_noise(soup: BeautifulSoup) -> None:
    for selector in GLOBAL_REMOVE_SELECTORS:
        for el in soup.select(selector):
            el.decompose()


def pick_main_container(soup: BeautifulSoup):
    for sel in MAIN_SELECTORS:
        el = soup.select_one(sel)
        if el:
            return el
    return soup


def remove_ecommerce_noise(container) -> None:
    for sel in ECOM_REMOVE_SELECTORS:
        for el in container.select(sel):
            el.decompose()


def extract_table_lines(container) -> list[str]:
    out = []
    for tr in container.find_all("tr"):
        cells = [element_text(c) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        if len(cells) >= 2:
            out.append(" | ".join(cells))
    return out


def extract_definition_lines(container) -> list[str]:
    out = []
    for dt in container.find_all("dt"):
        dd = dt.find_next_sibling("dd")
        k = element_text(dt)
        v = element_text(dd) if dd else ""
        if k and v:
            out.append(f"{k}: {v}")
    return out


def extract_price_points_stock_from_html(html: str) -> tuple[str, str, str]:
    visible = BeautifulSoup(html, "lxml").get_text(" ", strip=True)
    visible = re.sub(r"\s+", " ", visible)

    price = ""
    points = ""
    stock = ""

    m_price = PRICE_RE.search(visible)
    if m_price:
        price = m_price.group(1).strip()

    m_points = POINTS_RE.search(visible)
    if m_points:
        pts = re.sub(r"\s+", " ", m_points.group(1)).strip()
        points = f"{pts} points merci"

    m_stock = STOCK_RE.search(visible)
    if m_stock:
        stock = m_stock.group(1)

    return price, points, stock


# =========================
# Extractors
# =========================
def extract_product_hero_lines(container) -> list[str]:
    parts = []

    for sel in PRODUCT_HERO_SELECTORS:
        for el in container.select(sel):
            txt = element_text(el)
            if txt:
                parts.append(txt)

    for el in container.select("div.description-short li, div[id^='product-description-short-'] li"):
        txt = element_text(el)
        if txt:
            parts.append(txt)

    for el in container.select(".current-price, .price, [class*='price'], [id*='price'], .wbpdtbn, .product-actions"):
        txt = element_text(el)
        if txt:
            m = PRICE_RE.search(txt)
            if m:
                parts.append(f"Prix: {m.group(1)}")

    return clean_blocks(parts)


def extract_product_specs_lines(container) -> list[str]:
    parts = []

    blocks = []
    for sel in PRODUCT_KEEP_SELECTORS:
        blocks.extend(container.select(sel))
    if not blocks:
        blocks = [container]

    for block in blocks:
        for bad in block.select(".services-eshopping, .eshop-service-block, .review, .reviews, [id*='review'], [class*='review']"):
            bad.decompose()

        for el in block.find_all(["h2", "h3", "h4", "p", "li", "td", "th", "dt", "dd", "strong", "span"]):
            t = element_text(el)
            if t:
                parts.append(t)

        parts.extend(extract_table_lines(block))
        parts.extend(extract_definition_lines(block))

    return clean_blocks(parts)


def extract_product_details_page(soup, container) -> str:
    remove_ecommerce_noise(container)

    hero = extract_product_hero_lines(container)
    specs = extract_product_specs_lines(container)

    parts = clean_blocks(hero + specs)

    price, points, stock = extract_price_points_stock_from_html(str(soup))
    if price:
        parts.append(f"Prix: {price}")
    if points:
        parts.append(f"Points Merci: {points}")
    if stock:
        parts.append(f"Stock: {stock}")

    text = normalize_txt("\n".join(parts))

    if len(text.split()) < 8:
        text = to_embedding_text(container.get_text("\n", strip=True))

    return text


def extract_category_page(container) -> str:
    headline_selectors = [
        "h1", "h2", ".category-title", ".page-title",
        ".category-links a", ".service-card a", ".tabs a", ".tab a",
        ".carousel a", ".swiper a", ".owl-item a", ".card a",
    ]

    out = []
    seen = set()

    for sel in headline_selectors:
        for el in container.select(sel):
            txt = element_text(el)
            if not txt:
                continue
            low = txt.lower()
            if len(txt.split()) <= 12 and not re.search(r"\b\d+[.,]?\d*\b", txt):
                if low not in seen:
                    seen.add(low)
                    out.append(txt)

    if len(out) < 3:
        stop_words = {"description", "activation", "tarification", "pricing", "présentation", "presentation", "assistance", "faq"}
        lines = [normalize_txt(l) for l in container.get_text("\n", strip=True).split("\n")]
        lines = [l for l in lines if l]

        for line in lines:
            low = line.lower()
            if low in stop_words:
                break
            if len(line.split()) <= 10 and not re.search(r"\b\d+[.,]?\d*\b", line):
                if low not in seen:
                    seen.add(low)
                    out.append(line)
            if len(out) >= 12:
                break

    return normalize_txt(" | ".join(out[:12]))


def extract_listing_page(container) -> str:
    lines = []

    # Title
    t_el = container.select_one("h1, .page-title, .category-title")
    if t_el:
        title = element_text(t_el)
        if title:
            lines.append(title)

    # Count line
    visible = re.sub(r"\s+", " ", container.get_text(" ", strip=True))
    m_full = re.search(r"(il y a\s+\d+\s+produits?\.?|there are\s+\d+\s+products?\.?)", visible, re.I)
    if m_full:
        lines.append(m_full.group(1))
    else:
        m_total = container.select_one(".total-products, .products-counter, .category-count, .heading-counter")
        if m_total:
            t = element_text(m_total)
            if t:
                lines.append(t)
        else:
            m = COUNT_RE.search(visible)
            if m:
                lines.append(m.group(0))

    # Better root for product list
    root = None
    for sel in LISTING_CONTAINER_SELECTORS:
        root = container.select_one(sel)
        if root:
            break
    if root is None:
        root = container

    # Remove filter/sort blocks
    for bad in root.select(
        "#search_filters, #search_filters_wrapper, .facet, .facets, .left-column, "
        ".sort-by-row, .products-sort-order, [class*='filter'], [id*='filter'], [class*='sort']"
    ):
        bad.decompose()

    cards = []
    for sel in LISTING_CARD_SELECTORS:
        cards.extend(root.select(sel))

    extracted = set()

    if cards:
        for card in cards:
            name = ""
            price = ""
            stock = ""

            name_el = card.select_one(".product-title, .product-name, h2, h3, a[title], a.product-name")
            if name_el:
                name = element_text(name_el)

            if not name:
                for a in card.select("a[href]"):
                    href = a.get("href", "")
                    txt = element_text(a)
                    if PRODUCT_LINK_RE.search(href) and txt and len(txt.split()) >= 2:
                        name = txt
                        break

            if not name:
                continue

            blob = card.get_text(" ", strip=True)

            pm = PRICE_RE.search(blob)
            if pm:
                price = pm.group(1)

            sm = STOCK_RE.search(blob)
            if sm:
                stock = sm.group(1)

            line = name
            if price:
                line += f" | Prix: {price}"
            if stock:
                line += f" | Stock: {stock}"

            key = line.lower().strip()
            if key not in extracted:
                extracted.add(key)
                lines.append(line)

    # fallback: product names from links
    if len(extracted) < 3:
        for a in root.select("a[href]"):
            href = a.get("href", "")
            txt = element_text(a)
            if PRODUCT_LINK_RE.search(href) and txt and len(txt.split()) >= 2:
                key = txt.lower().strip()
                if key not in extracted:
                    extracted.add(key)
                    lines.append(txt)

    lines = clean_blocks(lines)
    return normalize_txt("\n".join(lines))


def extract_generic_page(container) -> str:
    parts = [element_text(el) for el in container.find_all(TEXT_TAGS)]
    parts.extend(extract_table_lines(container))
    parts.extend(extract_definition_lines(container))
    parts = clean_blocks(parts)
    return normalize_txt("\n".join(parts))


def extract_by_page_type(soup: BeautifulSoup, page_type: str) -> tuple[str, str, str]:
    title = get_title(soup)
    remove_global_noise(soup)
    container = pick_main_container(soup)

    raw_text = normalize_txt(container.get_text("\n", strip=True))

    if page_type == "product_page":
        clean_text = extract_product_details_page(soup, container)
    elif page_type == "category_page":
        clean_text = extract_category_page(container)
    elif page_type in {"listing_page", "brand_page"}:
        clean_text = extract_listing_page(container)
    else:
        clean_text = extract_generic_page(container)

    return title, raw_text, clean_text


# =========================
# Quality + dedupe
# =========================
def is_low_quality(text: str, page_type: str) -> tuple[bool, str]:
    words = text.split()
    alpha_ratio = sum(ch.isalpha() for ch in text) / max(1, len(text))

    min_words_by_type = {
        "product_page": 6,
        "listing_page": 2,
        "brand_page": 2,
        "category_page": 2,
        "landing_page": 10,
        "content_page": 20,
        "shop_page": 15,
        "other": 10,
    }
    min_words = min_words_by_type.get(page_type, 10)

    if len(words) < min_words:
        return True, f"Insufficient content ({len(words)}<{min_words})"

    min_alpha = 0.03 if page_type in {"product_page", "listing_page", "brand_page", "category_page"} else 0.15
    if alpha_ratio < min_alpha:
        return True, f"Low alphabetic ratio ({alpha_ratio:.2f}<{min_alpha})"

    return False, ""


def dedupe_records(records: list[Record]) -> list[Record]:
    seen = set()
    out = []
    for r in records:
        key = (r.language, r.content_hash)
        if not r.content_hash or key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out


def finalize_embedding_text(page_type: str, clean_text: str) -> str:
    return to_embedding_text(clean_text)


# =========================
# Async scraping
# =========================
async def fetch_soup(url: str, session: aiohttp.ClientSession, timeout_s: float = 12.0):
    headers = {"User-Agent": "Mozilla/5.0 (compatible; TextExtractor/5.1)"}
    async with session.get(url, headers=headers, timeout=aiohttp.ClientTimeout(total=timeout_s)) as resp:
        html = await resp.text()
        return BeautifulSoup(html, "lxml"), resp.status


async def extract_content_async(url: str, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore) -> Record:
    normalized_url = normalize_url(url)
    section, language = classify_url(url)
    page_type = classify_page_type(url)

    async with semaphore:
        try:
            soup, status_code = await fetch_soup(url, session)

            if is_soft_404(soup):
                return Record(
                    normalized_url, "", status_code, section, language, page_type,
                    "", "", "", "", "soft_404", "Soft 404"
                )

            title, raw_text, clean_text = extract_by_page_type(soup, page_type)
            embedding_text = finalize_embedding_text(page_type, clean_text)

            if page_type == "product_page" and len((embedding_text or "").split()) < 6:
                embedding_text = to_embedding_text(raw_text)

            low_quality, reason = is_low_quality(embedding_text, page_type)
            if low_quality:
                return Record(
                    normalized_url, title, status_code, section, language, page_type,
                    raw_text, clean_text, "", "", "error", reason
                )

            return Record(
                normalized_url, title, status_code, section, language, page_type,
                raw_text, clean_text, embedding_text, make_hash(embedding_text), "success", ""
            )

        except asyncio.TimeoutError:
            return Record(normalized_url, "", 0, section, language, page_type, "", "", "", "", "error", "Timeout")
        except aiohttp.ClientError as e:
            return Record(normalized_url, "", 0, section, language, page_type, "", "", "", "", "error", f"Network: {type(e).__name__}")
        except Exception as e:
            return Record(normalized_url, "", 0, section, language, page_type, "", "", "", "", "error", f"{type(e).__name__}: {str(e)[:120]}")





In [9]:
from collections import defaultdict
async def build_dataset_async(urls: list, output_file: str = "semantic_dataset.json", max_concurrent: int = 8):
    semaphore = asyncio.Semaphore(max_concurrent)
    records = []
    processed = 0

    connector = aiohttp.TCPConnector(limit_per_host=4, limit=80, ssl=False)
    timeout = aiohttp.ClientTimeout(total=18, connect=10, sock_read=12)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = [extract_content_async(url, session, semaphore) for url in urls]
        for task in asyncio.as_completed(tasks):
            rec = await task
            records.append(rec)
            processed += 1
            if processed % 50 == 0:
                print(f"Processed: {processed}/{len(urls)}")

    success_records = [r for r in records if r.status == "success"]
    success_records = dedupe_records(success_records)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump([asdict(r) for r in success_records], f, indent=2, ensure_ascii=False)

    soft_404 = sum(1 for r in records if r.status == "soft_404")
    errors = sum(1 for r in records if r.status == "error")

    error_reasons = defaultdict(int)
    for r in records:
        if r.status == "error":
            error_reasons[r.reason] += 1

    print("\nDataset Building Complete\n")
    print(f"Saved Success: {len(success_records)}")
    print(f"Soft 404: {soft_404}")
    print(f"Errors: {errors}")

    print("\nError Breakdown:")
    for reason, count in sorted(error_reasons.items(), key=lambda x: x[1], reverse=True):
        print(f"  {reason}: {count}")

    print(f"\nSaved to: {output_file}\n")
    return records, success_records

In [10]:
import time
async def main_async():
    print(f"Loaded {len(list_final)} URLs\n")
    start = time.time()
    records, saved_records = await build_dataset_async(
        list_final,   # remove slice for full run
        output_file="semantic_dataset.json",
        max_concurrent=8
    )
    elapsed = time.time() - start
    print(f"Execution: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    return records, saved_records

records, saved_records = await main_async()


Loaded 1333 URLs

Processed: 50/1333
Processed: 100/1333
Processed: 150/1333
Processed: 200/1333
Processed: 250/1333
Processed: 300/1333
Processed: 350/1333
Processed: 400/1333
Processed: 450/1333
Processed: 500/1333
Processed: 550/1333
Processed: 600/1333
Processed: 650/1333
Processed: 700/1333
Processed: 750/1333
Processed: 800/1333
Processed: 850/1333
Processed: 900/1333
Processed: 950/1333
Processed: 1000/1333
Processed: 1050/1333
Processed: 1100/1333
Processed: 1150/1333
Processed: 1200/1333
Processed: 1250/1333
Processed: 1300/1333

Dataset Building Complete

Saved Success: 838
Soft 404: 157
Errors: 110

Error Breakdown:
  Insufficient content (0<20): 65
  Insufficient content (0<2): 13
  Insufficient content (15<20): 10
  Insufficient content (19<20): 3
  Insufficient content (16<20): 3
  Insufficient content (18<20): 3
  Insufficient content (7<20): 3
  Insufficient content (5<20): 2
  Insufficient content (17<20): 2
  Timeout: 2
  Insufficient content (4<6): 1
  Insufficient c

In [1]:
import json
from pathlib import Path

main_path = Path("semantic_dataset.json")
tariff_path = Path("roaming_country_tariffs_selenium.json")
partner_path = Path("roaming_partner_operators_selenium.json")
out_path = Path("semantic_dataset_merged.json")

main_data = json.loads(main_path.read_text(encoding="utf-8"))
tariff_data = json.loads(tariff_path.read_text(encoding="utf-8"))
partner_data = json.loads(partner_path.read_text(encoding="utf-8"))

# optional: keep only successful records
main_data = [x for x in main_data if x.get("status") == "success"]
tariff_data = [x for x in tariff_data if x.get("status") == "success"]
partner_data = [x for x in partner_data if x.get("status") == "success"]

merged = main_data + tariff_data + partner_data

out_path.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Main dataset: {len(main_data)}")
print(f"Roaming tariffs: {len(tariff_data)}")
print(f"Roaming partner operators: {len(partner_data)}")
print(f"Total merged: {len(merged)}")
print(f"Saved to: {out_path}")

Main dataset: 819
Roaming tariffs: 308
Roaming partner operators: 324
Total merged: 1451
Saved to: semantic_dataset_merged.json
